In [1]:
import torch
from transformers import AutoModel, AutoFeatureExtractor

# 1. Проверяем и выбираем устройство
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model = AutoModel.from_pretrained("labhamlet/wavjepa-base", trust_remote_code=True).to(device)
if not hasattr(model, "all_tied_weights_keys"):
    model.all_tied_weights_keys = {}

extractor = AutoFeatureExtractor.from_pretrained("labhamlet/wavjepa-base", trust_remote_code=True)

audio = torch.zeros([1, 160000]).to(device)  # заглушка, 10 сек при 16кГц
extracted = extractor(audio, return_tensors="pt")
audio_feature = extracted['input_values']
result = model(audio_feature)
print(result[0].shape)
print(result[1].shape)

Loading weights:   0%|          | 0/457 [00:00<?, ?it/s]

/Users/vsevolod/.cache/huggingface/modules/transformers_modules/labhamlet/wavjepa_hyphen_base/6be4a5093f6c9adbbd55e00b6e5b8f067aa03345/feature_extraction_wavjepa.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  audio = torch.tensor(audio)


torch.Size([1, 996, 768])
torch.Size([1, 996])


/Users/vsevolod/Desktop/prosodic/lib/python3.14/site-packages/torch/masked/maskedtensor/creation.py:20: UserWarning: The PyTorch API of MaskedTensors is in prototype stage and will change in the near future. Please open a Github issue for features requests and see our documentation on the torch.masked module for further information about the project.
  return MaskedTensor(data, mask, requires_grad)


In [2]:
import torch
 
 
def inspect_and_pool(result):
    data, ts = result[0], result[1]   # data: [B, T, 768], ts: [B, T] -- timestamps в мс
 
    print("data.shape:", data.shape)
    print("ts.shape:", ts.shape)
    print("ts dtype:", ts.dtype)
    print("ts[0, :5] (первые 5 таймстемпов, мс):", ts[0, :5].tolist())
    print("ts[0, -5:] (последние 5 таймстемпов, мс):", ts[0, -5:].tolist())
 
    # Шаг между фреймами (должен быть ~10.036 мс для этой модели)
    step = (ts[0, 1] - ts[0, 0]).item()
    print(f"шаг между фреймами: {step:.3f} мс")
 
    # Простой mean pooling -- корректен, т.к. паддинг уже обрезан моделью (см. docstring)
    pooled = data.mean(dim=1)  # [B, 768]
    print("pooled (mean) shape:", pooled.shape)
    print("pooled vector norm:", pooled.norm(dim=-1))
 
    return pooled, ts
 
 
def get_embedding_at_time(data, ts, time_ms):
    """
    Найти эмбеддинг ближайшего фрейма к заданному моменту времени (в мс).
    Полезно для sliding-window анализа (например, сопоставить F0 в конкретный момент
    с локальным эмбеддингом, а не только с усреднённым по всему сегменту).
    """
    idx = (ts[0] - time_ms).abs().argmin().item()
    actual_ts = ts[0, idx].item()
    return data[:, idx, :], actual_ts, idx
 
 
if __name__ == "__main__":
    # Пример с фиктивными данными (шаг ~10.036 мс, как в реальной модели)
    B, T, D = 1, 389, 768
    step_ms = 10.036
    fake_data = torch.randn(B, T, D)
    fake_ts = torch.arange(T).float().unsqueeze(0) * step_ms  # [1, T]
 
    pooled, ts = inspect_and_pool((fake_data, fake_ts))
 
    print("\n--- Проверка сопоставления фрейма с моментом времени ---")
    emb, actual_ts, idx = get_embedding_at_time(fake_data, fake_ts, time_ms=2000.0)
    print(f"Запрошено ~2000 мс -> ближайший фрейм idx={idx}, реальный timestamp={actual_ts:.1f} мс")
 
    print("\n--- Итоговый pooled эмбеддинг для speaker-baseline (Блок 2) ---")
    print(pooled.shape)

data.shape: torch.Size([1, 389, 768])
ts.shape: torch.Size([1, 389])
ts dtype: torch.float32
ts[0, :5] (первые 5 таймстемпов, мс): [0.0, 10.03600025177002, 20.07200050354004, 30.108001708984375, 40.14400100708008]
ts[0, -5:] (последние 5 таймстемпов, мс): [3853.82421875, 3863.860107421875, 3873.89599609375, 3883.93212890625, 3893.968017578125]
шаг между фреймами: 10.036 мс
pooled (mean) shape: torch.Size([1, 768])
pooled vector norm: tensor([1.4452])

--- Проверка сопоставления фрейма с моментом времени ---
Запрошено ~2000 мс -> ближайший фрейм idx=199, реальный timestamp=1997.2 мс

--- Итоговый pooled эмбеддинг для speaker-baseline (Блок 2) ---
torch.Size([1, 768])


In [3]:
"""
Блок 2 -- ECAPA-TDNN speaker-эмбеддинги (baseline).

Источник: speechbrain/spkrec-ecapa-voxceleb (Apache-2.0)
Обучена на VoxCeleb1+VoxCeleb2, лосс -- AAM-Softmax (Additive Angular Margin),
один центр (прототип) на диктора -- это и есть "точечный" baseline,
с которым позже будем сравнивать sub-center / вариационный подход.
"""

import torch
import torchaudio


def load_ecapa():
    """Загрузить предобученную ECAPA-TDNN модель.
    
    ВАЖНО: SpeechBrain пока плохо работает с MPS, используем CPU.
    Для инференса это приемлемо — модель лёгкая, скорость адекватная.
    """
    from speechbrain.inference.speaker import EncoderClassifier

    # SpeechBrain имеет баг с MPS device_type — используем CPU
    model = EncoderClassifier.from_hparams(
        source="speechbrain/spkrec-ecapa-voxceleb",
        savedir="pretrained_models/spkrec-ecapa-voxceleb",
        run_opts={"device": "cpu"},  # Принудительно CPU из-за бага SpeechBrain с MPS
    )
    return model


def load_audio(path, target_sr=16000):
    """Та же предобработка что в Блоке 1: 16 кГц, моно"""
    waveform, sr = torchaudio.load(path)
    if sr != target_sr:
        waveform = torchaudio.transforms.Resample(sr, target_sr)(waveform)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    return waveform  # [1, N_samples]


def extract_embedding(model, waveform):
    """Извлечение speaker embedding с явной проверкой формы выхода"""
    # SpeechBrain на CPU, поэтому данные тоже должны быть на CPU
    waveform = waveform.cpu()
    
    with torch.no_grad():
        emb = model.encode_batch(waveform)

    print("Сырая форма выхода encode_batch:", emb.shape)

    # encode_batch обычно возвращает [batch, 1, D]
    if emb.dim() == 3 and emb.shape[1] == 1:
        emb = emb.squeeze(1)
        print("Убрана размерность-заглушка -> новая форма:", emb.shape)
    else:
        print("ПРЕДУПРЕЖДЕНИЕ: форма выхода не [B, 1, D]")

    return emb  # [B, D]


def sanity_checks(emb, expected_dim=192):
    """Быстрые проверки валидности эмбеддинга"""
    print("\n=== Sanity checks ===")

    dim = emb.shape[-1]
    print(f"Размерность: {dim} (ожидалось {expected_dim})")
    if dim != expected_dim:
        print(f"  !! Размерность отличается от ожидаемой")

    norm = emb.norm(dim=-1).item()
    print(f"Норма: {norm:.4f}")
    if norm < 1e-3:
        print("  !! Эмбеддинг почти нулевой")

    has_nan = torch.isnan(emb).any().item()
    has_inf = torch.isinf(emb).any().item()
    print(f"NaN: {has_nan}, Inf: {has_inf}")
    if has_nan or has_inf:
        print("  !! Критично: эмбеддинг повреждён")

    return {"dim_ok": dim == expected_dim, "norm": norm, "has_nan": has_nan, "has_inf": has_inf}


# === БЫСТРЫЙ ТЕСТ ===
print("="*60)
print("БЛОК 2: Загрузка ECAPA-TDNN и обработка test_record.wav")
print("="*60 + "\n")

print("Используемое устройство: CPU (SpeechBrain не полностью поддерживает MPS)\n")

# Загрузка модели
print("Загружаю ECAPA-TDNN (при первом запуске — скачивание весов)...")
model = load_ecapa()
print("✓ Модель загружена\n")

# Обработка файла
audio_path = "test_record.wav"
waveform = load_audio(audio_path)
print(f"✓ Загружен {audio_path}: {waveform.shape[1]/16000:.2f} сек\n")

# Извлечение эмбеддинга
emb = extract_embedding(model, waveform)

# Проверки
result = sanity_checks(emb)

print(f"\n{'='*60}")
print(f"БЛОК 2 завершён: baseline speaker embedding {emb.shape}")
print(f"{'='*60}")

БЛОК 2: Загрузка ECAPA-TDNN и обработка test_record.wav

Используемое устройство: CPU (SpeechBrain не полностью поддерживает MPS)

Загружаю ECAPA-TDNN (при первом запуске — скачивание весов)...
✓ Модель загружена

✓ Загружен test_record.wav: 3.90 сек

Сырая форма выхода encode_batch: torch.Size([1, 1, 192])
Убрана размерность-заглушка -> новая форма: torch.Size([1, 192])

=== Sanity checks ===
Размерность: 192 (ожидалось 192)
Норма: 353.3120
NaN: False, Inf: False

БЛОК 2 завершён: baseline speaker embedding torch.Size([1, 192])


In [ ]:
from transformers import PreTrainedModel

from .model import WavJEPA
from .configuration_wavjepa import WavJEPAConfig
from .audio_extractor import ConvFeatureExtractor
import torch 
from typing import Union 

class WavJEPAModel(PreTrainedModel):
    config_class = WavJEPAConfig

    def __init__(self, config):
        super().__init__(config)

        self.model = WavJEPA(
                feature_extractor = ConvFeatureExtractor(
                    conv_layers_spec = eval(config.extractor_config['conv_layers_spec']),
                    in_channels = config.extractor_config['in_channels'],
                    dropout = config.extractor_config['dropout'],
                    mode = config.extractor_config['mode'],
                    conv_bias = config.extractor_config['conv_bias'],
                    depthwise = config.extractor_config['depthwise'],
                    ),
                transformer_encoder_layers_cfg = config.encoder_layers_cfg,
                transformer_encoder_cfg = config.encoder_cfg,
                transformer_decoder_layers_cfg = config.decoder_layers_cfg,
                transformer_decoder_cfg = config.decoder_cfg,
                size = config.model_size,
        )
        self.post_init()

    def forward(self, tensor) -> Union[torch.Tensor, torch.Tensor]:
        return self.model.get_audio_representation(tensor)